# CHO AI: Zero-shot vs LoRA vs Full fine-tuning

**최종 배포: Full fine-tuning 350 step. 학습은 500 step까지, 그래프는 50 step 간격 전체 표시.**


### Colab 선택 패키지 충돌 수정


In [ ]:
import subprocess, sys
# No torchao quantization is used in this benchmark.
already_imported = any(name in sys.modules for name in ("torchao", "peft", "transformers", "chronos"))
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
# 확인한 공식 소스 커밋 고정. PyTorch는 Colab에 설치된 CUDA 버전을 유지합니다.
CHRONOS_COMMIT = "1224fafa91e1ecd6674f5204066b159e40b5e4a1"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/amazon-science/chronos-forecasting.git@" + CHRONOS_COMMIT,
    "peft", "accelerate", "pandas", "numpy", "matplotlib", "huggingface_hub"])

if already_imported:
    raise RuntimeError("설치 변경 완료. 런타임 > 세션 다시 시작 후 첫 셀부터 실행하세요. Drive 파일은 유지됩니다.")


## 1. Drive 경로와 학습 설정

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import json, shutil, hashlib, urllib.request, zipfile, re, io, platform
import numpy as np
import pandas as pd
import torch
from datetime import datetime, timezone

ROOT = Path("/content/drive/MyDrive/CHO_AI")  # 본인 경로: 이 부분 수정
ZIP_ON_DRIVE = ROOT / "raw" / "Dataset.zip"
AUTO_DOWNLOAD = False  # Drive에 원본이 없으면 True로 변경
SEED = 42
STEP_HOURS = 6
HORIZON = 4                 # 6시간 × 4 = 24시간 예측
CONTEXT = 48                # 최대 12일 이력; 짧은 이력도 사용
MIN_PAST = 8                # 최소 48시간 이력
NUM_STEPS = 500             # 50,100,...,500 step Validation 평가
BATCH_SIZE = 16             # 메모리 부족 시 8, 4 순으로 낮추기
LEARNING_RATE = 1e-5
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN = ROOT / "runs" / RUN_ID
LOCAL = Path("/content/cho_ai") / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Colab 런타임을 GPU로 변경하세요."
print(torch.cuda.get_device_name(0))
print("GPU GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("저장 경로:", RUN)

EVAL_EVERY = 50
# Scheduler is explicit but unchanged for the first monitoring run.
LR_SCHEDULER = "linear"
WARMUP_STEPS = 0
LATE_START_HOURS = 168  # Day 7; descriptive development definition, not a harvest limit

# Final choice agreed before this run's Test comparison.
FINAL_METHOD = "full"
FINAL_FULL_STEP = 350
assert NUM_STEPS >= FINAL_FULL_STEP and FINAL_FULL_STEP % EVAL_EVERY == 0


## 2. 원본 확인·로컬 복사


In [ ]:
if not ZIP_ON_DRIVE.exists():
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f"원본 Dataset.zip을 {ZIP_ON_DRIVE}에 넣거나 AUTO_DOWNLOAD=True로 설정하세요.")
    ZIP_ON_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    temp = ZIP_ON_DRIVE.with_suffix(".download")
    urllib.request.urlretrieve("https://zenodo.org/api/records/20829178/files/Dataset.zip/content", temp)
    temp.replace(ZIP_ON_DRIVE)

LOCAL_ZIP = LOCAL / "Dataset.zip"
shutil.copy2(ZIP_ON_DRIVE, LOCAL_ZIP)
def file_hash(path, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1024*1024), b""): h.update(b)
    return h.hexdigest()
assert file_hash(LOCAL_ZIP, "md5") == "71b2a35b7df2ed11644ec9f177453772", "원본 버전/파일 체크섬이 다릅니다."
DATA_SHA256 = file_hash(LOCAL_ZIP)
with zipfile.ZipFile(LOCAL_ZIP) as z:
    names = [n for n in z.namelist() if re.search(r"EXP\d+_(?:fed-batch|batch)_R\d+_(?:inline|offline)\.csv$", n)]
assert len(names) == 48, f"예상 CSV 48개, 실제 {len(names)}개"
print("CSV:", len(names), "SHA256:", DATA_SHA256)


## 3. 전처리 및 배포용 공통 코드


In [ ]:
CHO_MODULE = '"""CHO state forecast utilities. No Google Drive dependency in inference."""\nimport json, re, zipfile, io\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nTARGET_MAP = {\n    "VCD Mean (cells/mL)": "vcd_cells_ml",\n    "Viability Mean (%)": "viability_pct",\n    "Glucose (g/L)": "glucose_g_l",\n    "Lactate (g/L)": "lactate_g_l",\n    "Ammonium (mM)": "ammonium_mm",\n}\nCOV_MAP = {\n    "pH": "ph", "Dissolved O2 (% air sat)": "do_pct",\n    "Temperature (°C)": "temperature_c", "Liquid volume (L)": "volume_l",\n    "Permittivity (pF/cm)": "permittivity_pf_cm",\n    "Cumulative feed weight (g)": "feed_cumulative_g",\n}\nTARGETS = list(TARGET_MAP.values())\nCOVARIATES = list(COV_MAP.values())\nORIGIN = pd.Timestamp("2000-01-01")\n\ndef bin_table(raw, mapping, step_hours):\n    missing = set(mapping) | {"Time (h)"}\n    missing -= set(raw)\n    if missing: raise ValueError(f"Missing source columns: {sorted(missing)}")\n    t = pd.to_numeric(raw["Time (h)"], errors="coerce")\n    if t.isna().any() or (t < 0).any(): raise ValueError("Invalid elapsed time")\n    out = raw[list(mapping)].apply(pd.to_numeric, errors="coerce").rename(columns=mapping)\n    # ceil postpones availability to the bin end; never moves observations into the past.\n    out["time_h"] = np.ceil(t / step_hours).astype(int) * step_hours\n    out = out.sort_values("time_h", kind="stable").groupby("time_h").last()\n    return out\n\ndef prepare_zip(path, step_hours=6):\n    records, inventory = [], []\n    with zipfile.ZipFile(path) as z:\n        pat = re.compile(r"(EXP\\d+)_(fed-batch|batch)_(R\\d+)_offline\\.csv$")\n        for name in sorted(z.namelist()):\n            m = pat.search(name)\n            if not m: continue\n            exp, mode, reactor = m.groups()\n            read = lambda n: pd.read_csv(io.BytesIO(z.read(n)), encoding="utf-8-sig")\n            offline = bin_table(read(name), TARGET_MAP, step_hours)\n            inline_name = name.replace("_offline.csv", "_inline.csv")\n            inline = bin_table(read(inline_name), COV_MAP, step_hours)\n            # Use only the period for which offline follow-up exists.\n            end = int(offline.index.max())\n            grid = pd.Index(np.arange(0, end + 1, step_hours), name="time_h")\n            frame = offline.reindex(grid).join(inline.reindex(grid))\n            # Carry inline values for at most 12h. Targets are never filled here.\n            frame[COVARIATES] = frame[COVARIATES].ffill(limit=2)\n            frame = frame.reset_index()\n            frame["timestamp"] = ORIGIN + pd.to_timedelta(frame["time_h"], unit="h")\n            frame["item_id"] = exp + "_" + reactor\n            frame["experiment"] = exp\n            frame["mode"] = mode\n            records.append(frame)\n            inventory.append({"item_id": exp+"_"+reactor, "experiment": exp,\n                              "mode": mode, "offline_file": name, "inline_file": inline_name})\n    if len(records) != 24: raise ValueError(f"Expected 24 cultures, got {len(records)}")\n    return pd.concat(records, ignore_index=True), pd.DataFrame(inventory)\n\ndef model_frame(df):\n    return df[["item_id", "timestamp"] + TARGETS + COVARIATES].copy()\n\ndef forecast(pipeline, history, horizon=4, context=48):\n    # history must contain only observations available at the forecast cutoff.\n    frame = model_frame(history).sort_values(["item_id", "timestamp"])\n    result = pipeline.predict_df(frame, target=TARGETS, prediction_length=horizon,\n        quantile_levels=[0.1, 0.5, 0.9], context_length=context, batch_size=16,\n        cross_learning=False)\n    result.columns = [str(c) for c in result.columns]\n    return result\n\ndef backtest(pipeline, data, horizon=4, context=48, min_past=8):\n    rows = []\n    for item, group in data.groupby("item_id"):\n        group = group.sort_values("timestamp").reset_index(drop=True)\n        for cut in range(min_past, len(group)-horizon+1, horizon):\n            history, truth = group.iloc[:cut], group.iloc[cut:cut+horizon]\n            preds = forecast(pipeline, history, horizon, context)\n            last = history[TARGETS].ffill().iloc[-1]\n            for target in TARGETS:\n                p = preds[preds.target_name == target].set_index("timestamp")\n                for row in truth.itertuples(index=False):\n                    y = getattr(row, target)\n                    if not np.isfinite(y) or not np.isfinite(last[target]): continue\n                    q = p.loc[row.timestamp]\n                    rows.append(dict(item_id=item, experiment=row.experiment, mode=row.mode,\n                        cutoff=str(history.timestamp.iloc[-1]), timestamp=str(row.timestamp),\n                        target=target, y=float(y), persistence=float(last[target]),\n                        median=float(q["0.5"]), lower=float(q["0.1"]), upper=float(q["0.9"])))\n    if not rows: raise ValueError("No observed evaluation targets")\n    return pd.DataFrame(rows)\n\ndef metrics(predictions):\n    rows=[]\n    for (mode, target), g in predictions.groupby(["mode", "target"]):\n        for method in ["median", "persistence"]:\n            e = g[method]-g.y\n            rows.append(dict(mode=mode, target=target, method=method, n=len(g),\n                mae=float(e.abs().mean()), rmse=float(np.sqrt((e**2).mean())),\n                coverage_80=float(((g.y>=g.lower)&(g.y<=g.upper)).mean()) if method=="median" else None))\n    return pd.DataFrame(rows)\n\ndef residual_score(p):\n    # Heuristic prediction-deviation score, not a contamination probability.\n    return (p.y-p["median"]).abs() / np.maximum((p.upper-p.lower)/2, 1e-6)\n\ndef harvest_review(predictions, threshold_pct=80.0):\n    # Illustrative development trigger, not a validated harvest criterion.\n    v = predictions[predictions.target_name == "viability_pct"]\n    return {"status": "review" if (v["0.1"] < threshold_pct).any() else "no_trigger",\n            "illustrative_viability_threshold_pct": threshold_pct,\n            "optimal_harvest_day": None,\n            "reason": "Viability forecast only; titer and product-quality labels are unavailable."}\n\ndef load_predict(model_dir, history, horizon=None):\n    from chronos import Chronos2Pipeline\n    import torch\n    config = json.loads((Path(model_dir)/"preprocessing.json").read_text())\n    pipeline = Chronos2Pipeline.from_pretrained(str(model_dir),\n        device_map="cuda" if torch.cuda.is_available() else "cpu")\n    return forecast(pipeline, history, horizon or config["horizon"], config["context"])\n'
MODULE_PATH = LOCAL / "cho_model.py"
MODULE_PATH.write_text(CHO_MODULE, encoding="utf-8")
sys.path.insert(0, str(LOCAL))
from cho_model import *
data, inventory = prepare_zip(LOCAL_ZIP, STEP_HOURS)
display(inventory)
display(data.groupby("mode")[TARGETS].count())
data.to_csv(LOCAL / "prepared.csv", index=False)
inventory.to_csv(RUN / "source_inventory.csv", index=False)


## 4. 실험 단위 분리


In [ ]:
TRAIN_EXPS = [f"EXP{i:03d}" for i in [1,2,3,4,5,6,7,10]]
VAL_EXPS = ["EXP008", "EXP011"]
TEST_EXPS = ["EXP009", "EXP012"]
assert not (set(TRAIN_EXPS)&set(VAL_EXPS) or set(TRAIN_EXPS)&set(TEST_EXPS) or set(VAL_EXPS)&set(TEST_EXPS))
train = data[data.experiment.isin(TRAIN_EXPS)].copy()
val = data[data.experiment.isin(VAL_EXPS)].copy()
test = data[data.experiment.isin(TEST_EXPS)].copy()
split = {"train": TRAIN_EXPS, "validation": VAL_EXPS, "test": TEST_EXPS}
(RUN/"split.json").write_text(json.dumps(split, indent=2))
display(pd.crosstab(data.experiment, data["mode"]))


## 5. 세 모델의 공통 비교 설정



In [ ]:
import gc, time
from chronos import Chronos2Pipeline
from chronos.chronos2.preprocess import from_data_frame
from huggingface_hub import model_info, list_repo_files, hf_hub_download
from transformers import TrainerCallback, set_seed

BASE_MODEL = "amazon/chronos-2"  # 경량 실험은 autogluon/chronos-2-small; 전체 비교를 새 Run에서 실행
BASE_REVISION = model_info(BASE_MODEL).sha
METHODS = ["zero_shot", "lora", "full"]
TRAIN_LR = {"lora": LEARNING_RATE, "full": LEARNING_RATE}
MODEL_DIRS = {}
RESOURCE_ROWS = []
VAL_PREDS = {}
VAL_SCORES = {}
train_inputs = from_data_frame(model_frame(train), target_columns=TARGETS, prediction_length=HORIZON)
val_inputs = from_data_frame(model_frame(val), target_columns=TARGETS, prediction_length=HORIZON)

def cleanup_gpu():
    gc.collect(); torch.cuda.empty_cache()

def sync(): torch.cuda.synchronize()

def selection_score(p):
    ratios=[]
    for _, g in p.groupby("target"):
        denom=float(np.mean(np.abs(g.persistence-g.y)))
        if denom>1e-8: ratios.append(float(np.mean(np.abs(g["median"]-g.y)))/denom)
    if not ratios: raise ValueError("No usable validation denominators")
    return float(np.mean(ratios))

class DriveCheckpoint(TrainerCallback):
    def __init__(self, method): self.method=method
    def on_save(self, args, state, control, **kwargs):
        src=Path(args.output_dir)/f"checkpoint-{state.global_step}"
        if src.exists():
            dst=RUN/"checkpoints"/self.method/src.name
            shutil.copytree(src,dst,dirs_exist_ok=True)

class ParameterCounter(TrainerCallback):
    def __init__(self): self.total=0; self.trainable=0
    def on_train_begin(self,args,state,control,model=None,**kwargs):
        self.total=sum(p.numel() for p in model.parameters())
        self.trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)


### 5-A. 50 step 평가와 관계 분석 도구



In [ ]:
# Validation diagnostics; no Test data is used here.
import random
from contextlib import contextmanager
import matplotlib.pyplot as plt

def eval_detail(predictions, late_start_hours=168):
    p=predictions.copy()
    p["elapsed_h"]=(pd.to_datetime(p.timestamp)-pd.Timestamp("2000-01-01")).dt.total_seconds()/3600
    scopes=[("all",p)]+[(str(mode),g) for mode,g in p.groupby("mode")]
    rows=[]
    for scope,frame in scopes:
        for phase,g0 in [("all",frame),("late",frame[frame.elapsed_h>=late_start_hours])]:
            for target,g in g0.groupby("target"):
                if not len(g):continue
                e=g["median"]-g.y
                persist=float((g.persistence-g.y).abs().mean())
                rows.append(dict(scope=scope,phase=phase,target=target,n=len(g),
                    n_experiments=g.experiment.nunique(),mae=float(e.abs().mean()),
                    rmse=float(np.sqrt((e**2).mean())),bias=float(e.mean()),
                    persistence_mae=persist,
                    normalized_mae=float(e.abs().mean())/persist if persist>1e-8 else np.nan,
                    coverage_80=float(((g.y>=g.lower)&(g.y<=g.upper)).mean()),
                    mean_interval_width=float((g.upper-g.lower).mean())))
    return pd.DataFrame(rows)

def safe_corr(x,y,rank=False):
    q=pd.DataFrame({"x":x,"y":y}).replace([np.inf,-np.inf],np.nan).dropna()
    if len(q)<4 or q.x.nunique()<2 or q.y.nunique()<2:return np.nan
    if rank:q=q.rank(method="average")
    return float(q.x.corr(q.y))

def relationship_summary(p):
    # Match the two targets on identical reactor/cutoff/timestamp, never across reactors.
    keys=["item_id","experiment","mode","cutoff","timestamp"]
    cols=keys+["y","median"]
    v=p[p.target=="vcd_cells_ml"][cols].rename(columns={"y":"actual_vcd","median":"pred_vcd"})
    a=p[p.target=="viability_pct"][cols].rename(columns={"y":"actual_viability","median":"pred_viability"})
    pairs=v.merge(a,on=keys,validate="one_to_one")
    rows=[]
    for item,g in pairs.groupby("item_id"):
        rows.append(dict(item_id=item,experiment=g.experiment.iloc[0],mode=g["mode"].iloc[0],n=len(g),
            actual_pearson=safe_corr(g.actual_vcd,g.actual_viability),
            actual_spearman=safe_corr(g.actual_vcd,g.actual_viability,True),
            predicted_pearson=safe_corr(g.pred_vcd,g.pred_viability),
            signed_error_pearson=safe_corr(g.pred_vcd-g.actual_vcd,g.pred_viability-g.actual_viability)))
    return pairs,pd.DataFrame(rows)

def plot_predictions(p,path,title):
    items=sorted(p.item_id.unique())
    fig,axes=plt.subplots(len(items),2,figsize=(13,3*len(items)),squeeze=False)
    for row,item in enumerate(items):
        for col,target in enumerate(["vcd_cells_ml","viability_pct"]):
            g=p[(p.item_id==item)&(p.target==target)].copy().sort_values("timestamp")
            ax=axes[row,col]
            if not len(g):continue
            t=(pd.to_datetime(g.timestamp)-pd.Timestamp("2000-01-01")).dt.total_seconds()/86400
            scale=1e6 if target=="vcd_cells_ml" else 1
            ax.plot(t,g.y/scale,"ko-",label="Observed",markersize=3)
            ax.plot(t,g["median"]/scale,label="Predicted")
            ax.fill_between(t.to_numpy(),g.lower.to_numpy()/scale,g.upper.to_numpy()/scale,alpha=.2,label="P10-P90")
            ax.set_title(item+" / "+target);ax.set_xlabel("Culture day")
            ax.set_ylabel("VCD (million cells/mL)" if col==0 else "Viability (%)")
            ax.legend(fontsize=8)
    fig.suptitle(title);fig.tight_layout();fig.savefig(path,dpi=140);plt.close(fig)

@contextmanager
def preserve_training_state(model):
    # Monitoring must not alter training RNG streams or per-module train/eval state.
    py=random.getstate();np_state=np.random.get_state();cpu=torch.get_rng_state()
    cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    flags=[(m,m.training) for m in model.modules()]
    try:
        model.eval()
        with torch.inference_mode():yield
    finally:
        for m,flag in flags:m.training=flag
        random.setstate(py);np.random.set_state(np_state);torch.set_rng_state(cpu)
        if cuda is not None:torch.cuda.set_rng_state_all(cuda)

class StepValidationMonitor(TrainerCallback):
    def __init__(self,method):
        self.method=method;self.losses={};self.done=set();self.seconds=0
    def on_evaluate(self,args,state,control,metrics=None,**kwargs):
        self.losses[int(state.global_step)]=(metrics or {}).get("eval_loss",np.nan)
    def on_save(self,args,state,control,model=None,optimizer=None,**kwargs):
        step=int(state.global_step)
        if step in self.done:return
        self.done.add(step)
        src=Path(args.output_dir)/f"checkpoint-{step}"
        if not src.is_dir():raise FileNotFoundError(src)
        # Retain every evaluated checkpoint even when Trainer deletes old local checkpoints.
        dst=RUN/"checkpoints"/self.method/src.name
        shutil.copytree(src,dst,dirs_exist_ok=True)
        started=time.perf_counter()
        with preserve_training_state(model):
            pred=backtest(Chronos2Pipeline(model=model),val,HORIZON,CONTEXT,MIN_PAST)
        sync();self.seconds+=time.perf_counter()-started
        lr=float(optimizer.param_groups[0]["lr"]) if optimizer is not None else np.nan
        record_validation(self.method,step,pred,self.losses.get(step,np.nan),lr)
        print(f"[{self.method}] step {step}: variable-level Validation saved")

STEP_ROWS=[]
def record_validation(method,step,pred,eval_loss=np.nan,learning_rate=np.nan):
    folder=RUN/"monitoring"/method/f"step-{step:04d}"
    folder.mkdir(parents=True,exist_ok=True)
    pred.to_csv(folder/"predictions.csv",index=False)
    detail=eval_detail(pred,LATE_START_HOURS)
    detail["method"]=method;detail["step"]=step
    detail["eval_loss"]=eval_loss;detail["learning_rate"]=learning_rate
    detail.to_csv(folder/"metrics.csv",index=False)
    STEP_ROWS.extend(detail.to_dict("records"))
    pd.DataFrame(STEP_ROWS).to_csv(RUN/"validation_step_metrics.csv",index=False)
    pairs,relations=relationship_summary(pred)
    pairs.to_csv(folder/"vcd_viability_pairs.csv",index=False)
    relations.to_csv(folder/"vcd_viability_prediction_relationship.csv",index=False)
    plot_predictions(pred,folder/"vcd_viability_predictions.png",f"{method}: step {step} Validation")
    display(detail[(detail.phase=="all")&(detail.scope=="fed-batch")][
        ["method","step","target","mae","rmse","coverage_80","mean_interval_width","eval_loss","learning_rate"]])

def checkpoint_choice(rows,reference):
    # Guard both overall and Fed-batch viability MAE against zero-shot.
    candidates=[]
    for step,g in rows.groupby("step"):
        vals={};eligible=True
        for scope in ["all","fed-batch"]:
            q=g[(g.scope==scope)&(g.phase=="all")&(g.target=="viability_pct")]
            ref=reference[(reference.scope==scope)&(reference.phase=="all")&(reference.target=="viability_pct")]
            if len(q)!=1 or len(ref)!=1:raise ValueError("Missing viability selection metric")
            vals[scope]=float(q.mae.iloc[0]);eligible &= vals[scope]<=float(ref.mae.iloc[0])+1e-12
        agg=g[(g.scope=="all")&(g.phase=="all")].normalized_mae.mean()
        candidates.append(dict(step=int(step),eligible=bool(eligible),
            viability_mae=vals["all"],fed_batch_viability_mae=vals["fed-batch"],normalized_mae=float(agg)))
    table=pd.DataFrame(candidates)
    pool=table[table.eligible] if table.eligible.any() else table
    best=pool.sort_values(["viability_mae","normalized_mae","step"]).iloc[0].to_dict()
    return best,table


In [ ]:
# Raw observed VCD/viability pairs; Train + Validation only, actual sampling times.
def raw_relationship_data(zip_path,allowed_experiments):
    frames=[]
    with zipfile.ZipFile(zip_path) as z:
        for name in sorted(z.namelist()):
            m=re.search(r"(EXP\d+)_(fed-batch|batch)_(R\d+)_offline\.csv$",name)
            if not m or m.group(1) not in allowed_experiments:continue
            exp,mode,reactor=m.groups()
            a=pd.read_csv(io.BytesIO(z.read(name)),encoding="utf-8-sig")
            a=a[["Time (h)","VCD Mean (cells/mL)","TCD Mean (cells/mL)","Viability Mean (%)"]].copy()
            a.columns=["time_h","vcd","tcd","viability"]
            a=a.apply(pd.to_numeric,errors="coerce").sort_values("time_h")
            a["item_id"]=exp+"_"+reactor;a["experiment"]=exp;a["mode"]=mode
            a["phase"]=np.select([a.time_h<72,a.time_h<LATE_START_HOURS],["early_0_72h","middle"],default="late")
            dt=a.time_h.diff().where(lambda v:v>0)
            a["vcd_change_per_h"]=a.vcd.diff()/dt
            a["viability_change_per_h"]=a.viability.diff()/dt
            a["ratio_of_mean_counts_pct"]=100*a.vcd/a.tcd.where(a.tcd>0)
            frames.append(a)
    return pd.concat(frames,ignore_index=True)

def raw_correlations(raw):
    rows=[]
    for item,g in raw.groupby("item_id"):
        for phase,a in [("all",g)]+list(g.groupby("phase")):
            rows.append(dict(item_id=item,experiment=a.experiment.iloc[0],mode=a["mode"].iloc[0],
                phase=phase,n=a[["vcd","viability"]].dropna().shape[0],
                pearson=safe_corr(a.vcd,a.viability),spearman=safe_corr(a.vcd,a.viability,True),
                change_rate_pearson=safe_corr(a.vcd_change_per_h,a.viability_change_per_h)))
    # Center within culture to reduce between-culture level confounding. Still descriptive.
    for mode,g in raw.groupby("mode"):
        q=g.dropna(subset=["vcd","viability"]).copy()
        q["vcd_centered"]=q.vcd-q.groupby("item_id").vcd.transform("mean")
        q["viability_centered"]=q.viability-q.groupby("item_id").viability.transform("mean")
        rows.append(dict(item_id="within_culture_centered",experiment="multiple",mode=mode,phase="all",
            n=len(q),pearson=safe_corr(q.vcd_centered,q.viability_centered),
            spearman=safe_corr(q.vcd_centered,q.viability_centered,True),
            change_rate_pearson=safe_corr(q.vcd_change_per_h,q.viability_change_per_h)))
    return pd.DataFrame(rows)

REL_DIR=RUN/"relationships";REL_DIR.mkdir(exist_ok=True)
raw_pairs=raw_relationship_data(LOCAL_ZIP,set(TRAIN_EXPS+VAL_EXPS))
raw_corr=raw_correlations(raw_pairs)
raw_pairs.to_csv(REL_DIR/"raw_vcd_viability_pairs.csv",index=False)
raw_corr.to_csv(REL_DIR/"raw_vcd_viability_correlations.csv",index=False)
display(raw_corr)
items=sorted(raw_pairs.item_id.unique())
fig,axes=plt.subplots(int(np.ceil(len(items)/4)),4,figsize=(15,3*int(np.ceil(len(items)/4))),squeeze=False)
for ax,item in zip(axes.flat,items):
    g=raw_pairs[raw_pairs.item_id==item]
    scatter=ax.scatter(g.vcd/1e6,g.viability,c=g.time_h/24,cmap="viridis",s=20)
    ax.set_title(item);ax.set_xlabel("VCD (million cells/mL)");ax.set_ylabel("Viability (%)")
    fig.colorbar(scatter,ax=ax,label="Culture day")
for ax in list(axes.flat)[len(items):]:ax.axis("off")
fig.tight_layout();fig.savefig(REL_DIR/"raw_vcd_viability_scatter.png",dpi=140);plt.show();plt.close(fig)
fig,axes=plt.subplots(int(np.ceil(len(items)/4)),4,figsize=(15,3*int(np.ceil(len(items)/4))),squeeze=False)
for ax,item in zip(axes.flat,items):
    g=raw_pairs[raw_pairs.item_id==item];right=ax.twinx()
    ax.plot(g.time_h/24,g.vcd/1e6,"o-",color="tab:blue",markersize=2)
    right.plot(g.time_h/24,g.viability,"s-",color="tab:orange",markersize=2)
    ax.set_title(item);ax.set_xlabel("Culture day");ax.set_ylabel("VCD (million/mL)",color="tab:blue")
    right.set_ylabel("Viability (%)",color="tab:orange")
for ax in list(axes.flat)[len(items):]:ax.axis("off")
fig.tight_layout();fig.savefig(REL_DIR/"raw_vcd_viability_timecourses.png",dpi=140);plt.show();plt.close(fig)
print("관측 관계는 인과관계나 학습 gradient 충돌을 입증하지 않습니다. p-value는 계산하지 않습니다.")
print("평균 VCD/평균 TCD 비율과 반복 측정 viability의 평균은 정확히 같지 않을 수 있습니다.")


## 6. 500 step 학습 · 50 step 모니터링 · 최종 Full 350 고정



In [ ]:
CHECKPOINT_CHOICES={}
for method in METHODS:
    cleanup_gpu();set_seed(SEED)
    base=Chronos2Pipeline.from_pretrained(BASE_MODEL,revision=BASE_REVISION,device_map="cuda")
    params=ParameterCounter();train_seconds=0.0;monitor=None;chosen_step=0
    torch.cuda.reset_peak_memory_stats()
    if method=="zero_shot":
        candidate=base
        params.total=sum(p.numel() for p in candidate.model.parameters());params.trainable=0
        vp=backtest(candidate,val,HORIZON,CONTEXT,MIN_PAST)
        record_validation(method,0,vp)
        ZERO_REFERENCE=eval_detail(vp,LATE_START_HOURS)
        CHECKPOINT_CHOICES[method]=dict(step=0,eligible=True)
        VAL_PREDS[method]=vp
    else:
        # Step 0 reference is the same untouched base for both methods.
        record_validation(method,0,VAL_PREDS["zero_shot"].copy())
        monitor=StepValidationMonitor(method)
        sync();started=time.perf_counter()
        candidate=base.fit(inputs=train_inputs,validation_inputs=val_inputs,
            prediction_length=HORIZON,context_length=CONTEXT,min_past=MIN_PAST,
            finetune_mode=method,learning_rate=TRAIN_LR[method],
            num_steps=NUM_STEPS,batch_size=BATCH_SIZE,
            output_dir=str(LOCAL/"training"/method),finetuned_ckpt_name="final",
            callbacks=[monitor,params],save_strategy="steps",save_steps=EVAL_EVERY,
            eval_strategy="steps",eval_steps=EVAL_EVERY,logging_steps=EVAL_EVERY,
            lr_scheduler_type=LR_SCHEDULER,warmup_steps=WARMUP_STEPS,
            load_best_model_at_end=False,report_to="none",seed=SEED,save_total_limit=2)
        sync();train_seconds=time.perf_counter()-started
    train_peak=torch.cuda.max_memory_allocated()/2**30 if method!="zero_shot" else None
    del candidate;del base;cleanup_gpu()
    if method!="zero_shot":
        rows=pd.DataFrame(STEP_ROWS)
        rows=rows[(rows.method==method)&(rows.step>0)]
        best,table=checkpoint_choice(rows,ZERO_REFERENCE)
        # Keep the automatically suggested checkpoint for transparency.
        best["automatic_suggestion_step"]=int(best["step"])
        if method=="full":
            fixed=table[table.step==FINAL_FULL_STEP]
            if len(fixed)!=1:raise RuntimeError("Full 350 checkpoint was not evaluated/saved")
            best=dict(fixed.iloc[0].to_dict(),automatic_suggestion_step=int(best["step"]),
                choice_reason="User-fixed Full 350 after prior Validation inspection")
        CHECKPOINT_CHOICES[method]=best;chosen_step=int(best["step"])
        table.to_csv(RUN/f"checkpoint_selection_{method}.csv",index=False)
        checkpoint=RUN/"checkpoints"/method/f"checkpoint-{chosen_step}"
        candidate=Chronos2Pipeline.from_pretrained(str(checkpoint),device_map="cuda")
    else:
        candidate=Chronos2Pipeline.from_pretrained(BASE_MODEL,revision=BASE_REVISION,device_map="cuda")
    if hasattr(candidate.model,"merge_and_unload"):
        candidate=Chronos2Pipeline(model=candidate.model.merge_and_unload())
    folder=LOCAL/"models"/method
    candidate.save_pretrained(folder)
    sample=val[val.item_id==sorted(val.item_id.unique())[0]].iloc[:MIN_PAST].copy()
    expected=forecast(candidate,sample,HORIZON,CONTEXT)
    del candidate;cleanup_gpu()
    candidate=Chronos2Pipeline.from_pretrained(str(folder),device_map="cuda")
    actual=forecast(candidate,sample,HORIZON,CONTEXT)
    assert np.allclose(expected[["0.1","0.5","0.9"]],actual[["0.1","0.5","0.9"]],rtol=1e-4,atol=1e-4)
    torch.cuda.reset_peak_memory_stats();sync();started=time.perf_counter()
    vp=backtest(candidate,val,HORIZON,CONTEXT,MIN_PAST)
    sync();val_seconds=time.perf_counter()-started
    VAL_PREDS[method]=vp;VAL_SCORES[method]=selection_score(vp)
    vp.to_csv(RUN/f"validation_{method}.csv",index=False)
    RESOURCE_ROWS.append(dict(method=method,total_parameters=params.total,trainable_parameters=params.trainable,
        trainable_pct=100*params.trainable/params.total,
        steps=0 if method=="zero_shot" else NUM_STEPS,selected_step=chosen_step,
        passes_viability_guard=bool(CHECKPOINT_CHOICES[method]["eligible"]),
        learning_rate=None if method=="zero_shot" else TRAIN_LR[method],train_seconds=train_seconds,
        monitoring_seconds=0.0 if monitor is None else monitor.seconds,
        train_peak_allocated_gib=train_peak,validation_seconds=val_seconds,
        validation_peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30,
        validation_normalized_mae=VAL_SCORES[method]))
    shutil.copytree(folder,RUN/"models"/method,dirs_exist_ok=True)
    MODEL_DIRS[method]=RUN/"models"/method
    pd.DataFrame(RESOURCE_ROWS).to_csv(RUN/"resources.csv",index=False)
    del candidate;cleanup_gpu()

def viability_key(method):
    d=eval_detail(VAL_PREDS[method],LATE_START_HOURS)
    q=d[(d.scope=="all")&(d.phase=="all")&(d.target=="viability_pct")]
    return (float(q.mae.iloc[0]),VAL_SCORES[method])
eligible=[m for m in METHODS if CHECKPOINT_CHOICES[m]["eligible"]]
AUTO_SELECTED=min(eligible,key=viability_key)
SELECTED=FINAL_METHOD
if not CHECKPOINT_CHOICES[SELECTED]["eligible"]:
    print("주의: 고정한 Full 350이 이번 Run의 viability 보호조건을 통과하지 못했습니다. 결과에 이를 표시합니다.")
# No silent fallback: deployment remains explicitly Full step 350.
assert CHECKPOINT_CHOICES["full"]["step"]==FINAL_FULL_STEP
(RUN/"deployment_selection.json").write_text(json.dumps(dict(
    method=SELECTED,step=FINAL_FULL_STEP,automatic_suggestion=AUTO_SELECTED,
    passes_viability_guard=bool(CHECKPOINT_CHOICES[SELECTED]["eligible"]),
    reason="User-fixed from prior Validation inspection; Test is exploratory"),indent=2))
(RUN/"checkpoint_choices.json").write_text(json.dumps(CHECKPOINT_CHOICES,indent=2))
print("Final deployment: Full step 350. Automatic suggestion:",AUTO_SELECTED)
print(CHECKPOINT_CHOICES)
display(pd.DataFrame(RESOURCE_ROWS))


In [ ]:
# Learning curves from fixed Validation windows, including zero-shot at step 0.
step_metrics=pd.DataFrame(STEP_ROWS)
fig,axes=plt.subplots(2,3,figsize=(15,8))
for method in ["lora","full"]:
    v=step_metrics[(step_metrics.method==method)&(step_metrics.scope=="fed-batch")&
        (step_metrics.phase=="all")&(step_metrics.target=="viability_pct")].sort_values("step")
    for ax,col in zip(axes[0],["mae","rmse","coverage_80"]):
        ax.plot(v.step,v[col],"o-",label=method);ax.set_title("Viability / "+col)
    axes[1,0].plot(v.step,v.mean_interval_width,"o-",label=method)
    axes[1,1].plot(v.step,v.eval_loss,"o-",label=method)
    axes[1,2].plot(v.step,v.learning_rate,"o-",label=method)
    for ax,col in zip(axes[0],["mae","rmse","coverage_80"]):
        z=ZERO_REFERENCE[(ZERO_REFERENCE.scope=="fed-batch")&(ZERO_REFERENCE.phase=="all")&
            (ZERO_REFERENCE.target=="viability_pct")][col].iloc[0]
        if method=="lora":ax.axhline(z,color="grey",linestyle="--",label="zero-shot")
axes[0,2].axhline(.8,color="black",linestyle=":",label="nominal 80%")
for ax,title in zip(axes[1],["Viability interval width (%p)","Overall eval_loss","Learning rate"]):ax.set_title(title)
for ax in axes.flat:ax.set_xlabel("Training step");ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(RUN/"validation_viability_learning_curves.png",dpi=140);plt.show();plt.close(fig)
fig,axes=plt.subplots(1,len(TARGETS),figsize=(19,4))
for ax,target in zip(axes,TARGETS):
    for method in ["lora","full"]:
        g=step_metrics[(step_metrics.method==method)&(step_metrics.scope=="fed-batch")&
            (step_metrics.phase=="all")&(step_metrics.target==target)].sort_values("step")
        ax.plot(g.step,g.mae,"o-",label=method)
    ax.set_title(target);ax.set_xlabel("Training step");ax.set_ylabel("MAE (native units)");ax.legend()
fig.tight_layout();fig.savefig(RUN/"validation_all_target_mae_curves.png",dpi=140);plt.show();plt.close(fig)
display(step_metrics[(step_metrics.scope=="fed-batch")&(step_metrics.phase=="late")&
    (step_metrics.target=="viability_pct")][["method","step","n","mae","rmse","coverage_80","mean_interval_width"]])


## 7. 세 모델의 공통 Test 평가



In [ ]:
TEST_PREDS={};DETAIL=[];TEST_TIMING=[]
for method in METHODS:
    cleanup_gpu()
    candidate=Chronos2Pipeline.from_pretrained(str(MODEL_DIRS[method]),device_map="cuda")
    sample=test[test.item_id==sorted(test.item_id.unique())[0]].iloc[:MIN_PAST].copy()
    forecast(candidate,sample,HORIZON,CONTEXT)  # warmup
    sync();torch.cuda.reset_peak_memory_stats();started=time.perf_counter()
    tp=backtest(candidate,test,HORIZON,CONTEXT,MIN_PAST)
    sync();elapsed=time.perf_counter()-started
    TEST_PREDS[method]=tp
    tp.to_csv(RUN/f"test_predictions_{method}.csv",index=False)
    windows=tp[["item_id","cutoff"]].drop_duplicates().shape[0]
    TEST_TIMING.append(dict(method=method,test_seconds=elapsed,
        ms_per_forecast_window=elapsed*1000/windows,
        test_peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30))
    del candidate;cleanup_gpu()

def compare_predictions(predictions):
    # All methods must use exactly the same observed targets.
    key=["item_id","cutoff","timestamp","target"]
    reference=predictions["zero_shot"].sort_values(key).reset_index(drop=True)
    records=[]
    for method,pred in predictions.items():
        pred=pred.sort_values(key).reset_index(drop=True)
        assert pred[key].equals(reference[key]), "Different evaluation rows"
        assert np.allclose(pred.y,reference.y), "Different ground truth"
        scopes=[("all",pred)]+[(m,g) for m,g in pred.groupby("mode")]
        for scope,frame in scopes:
            for target,g in frame.groupby("target"):
                err=g["median"]-g.y
                baseline=float(np.mean(np.abs(g.persistence-g.y)))
                mae=float(err.abs().mean())
                records.append(dict(method=method,scope=scope,target=target,n=len(g),
                    n_experiments=g.experiment.nunique(),mae=mae,
                    rmse=float(np.sqrt((err**2).mean())),persistence_mae=baseline,
                    normalized_mae=mae/baseline if baseline>1e-8 else np.nan,
                    coverage_80=float(((g.y>=g.lower)&(g.y<=g.upper)).mean()),
                    mean_interval_width=float((g.upper-g.lower).mean())))
    detail=pd.DataFrame(records)
    zero=detail[detail.method=="zero_shot"][["scope","target","mae"]].rename(columns={"mae":"zero_shot_mae"})
    detail=detail.merge(zero,on=["scope","target"],validate="many_to_one")
    detail["improvement_vs_zero_shot_pct"]=np.where(detail.zero_shot_mae>1e-8,
        100*(1-detail.mae/detail.zero_shot_mae),np.nan)
    return detail

detail=compare_predictions(TEST_PREDS)
detail.to_csv(RUN/"comparison_by_target.csv",index=False)
summary=detail.groupby(["method","scope"],as_index=False).agg(
    mean_normalized_mae=("normalized_mae","mean"),mean_coverage_80=("coverage_80","mean"))
summary=summary.merge(pd.DataFrame(RESOURCE_ROWS),on="method").merge(pd.DataFrame(TEST_TIMING),on="method")
summary["final_deployment_model"]=summary.method==SELECTED
summary.to_csv(RUN/"comparison_summary.csv",index=False)
display(summary)
display(detail[detail.scope=="fed-batch"])


## 8. 이탈 검토·수확 검토 신호



In [ ]:
THRESHOLDS={}
for method in METHODS:
    THRESHOLDS[method]={target:float(np.quantile(residual_score(g),0.99))
        for target,g in VAL_PREDS[method].groupby("target")}
    tp=TEST_PREDS[method].copy()
    tp["deviation_score"]=residual_score(tp)
    tp["deviation_flag"]=tp.deviation_score>tp.target.map(THRESHOLDS[method])
    tp.to_csv(RUN/f"test_deviation_{method}.csv",index=False)
candidate=Chronos2Pipeline.from_pretrained(str(MODEL_DIRS[SELECTED]),device_map="cuda")
sample=test[test.item_id==sorted(test.item_id.unique())[0]].iloc[:MIN_PAST].copy()
demo=forecast(candidate,sample,HORIZON,CONTEXT)
print(harvest_review(demo,threshold_pct=80.0))
del candidate;cleanup_gpu()


## 9. GitHub용 비교표·소스 패키지 + Hugging Face 모델 카드 생성


In [ ]:
def markdown_table(df):
    def fmt(x):
        if pd.isna(x): return "—"
        if isinstance(x,(float,np.floating)): return f"{x:.4f}"
        return str(x).replace("|","/").replace("\n"," ")
    head="| "+" | ".join(map(str,df.columns))+" |"
    rule="| "+" | ".join(["---"]*len(df.columns))+" |"
    return "\n".join([head,rule]+["| "+" | ".join(fmt(x) for x in row)+" |" for row in df.itertuples(index=False,name=None)])

REPO_DIR=RUN/"github_release"
(REPO_DIR/"results").mkdir(parents=True,exist_ok=True)
requirements="git+https://github.com/amazon-science/chronos-forecasting.git@"+CHRONOS_COMMIT+"\npeft\naccelerate\npandas\nnumpy\ntorch\nhuggingface_hub\n"
license_bytes=urllib.request.urlopen("https://raw.githubusercontent.com/amazon-science/chronos-forecasting/"+CHRONOS_COMMIT+"/LICENSE",timeout=30).read()
upstream={}
for name in list_repo_files(BASE_MODEL,revision=BASE_REVISION):
    if name.upper() in {"LICENSE","LICENSE.TXT","NOTICE","NOTICE.TXT"}:
        upstream["UPSTREAM_"+name]=Path(hf_hub_download(BASE_MODEL,name,revision=BASE_REVISION)).read_bytes()
environment=dict(python=platform.python_version(),torch=torch.__version__,cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),base_model=BASE_MODEL,base_revision=BASE_REVISION,
    chronos_commit=CHRONOS_COMMIT,data_sha256=DATA_SHA256,seed=SEED,
    step_hours=STEP_HOURS,horizon=HORIZON,context=CONTEXT,min_past=MIN_PAST,
    batch_size=BATCH_SIZE,num_steps=NUM_STEPS,learning_rates=TRAIN_LR,eval_every=EVAL_EVERY,
    scheduler=LR_SCHEDULER,warmup_steps=WARMUP_STEPS,late_start_hours=LATE_START_HOURS,checkpoint_choices=CHECKPOINT_CHOICES,final_method=FINAL_METHOD,final_full_step=FINAL_FULL_STEP)
freeze=subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)
for method in METHODS:
    folder=MODEL_DIRS[method]
    config=dict(environment,method=method,target_map=TARGET_MAP,covariate_map=COV_MAP,
        binning="ceil to right edge, last available",inline_forward_fill_limit=2,
        missing_targets="NaN; never fill for scoring",anomaly_thresholds=THRESHOLDS[method])
    (folder/"preprocessing.json").write_text(json.dumps(config,indent=2,ensure_ascii=False))
    (folder/"cho_model.py").write_text(CHO_MODULE)
    (folder/"requirements.txt").write_text(requirements)
    (folder/"LICENSE").write_bytes(license_bytes)
    for name,b in upstream.items(): (folder/name).write_bytes(b)
    (folder/"split.json").write_text(json.dumps(split,indent=2))
    (folder/"environment-lock.txt").write_text(freeze)
    detail[detail.method==method].to_csv(folder/"test_metrics.csv",index=False)
    card=f"""---
license: apache-2.0
base_model: {BASE_MODEL}
tags:
- time-series
- bioprocess
- cho
---
# CHO state forecasting: {method}
Base revision: {BASE_REVISION}. Method: {method}. Seed: {SEED}.
Exported checkpoint step: {CHECKPOINT_CHOICES[method]["step"]}.
Training budget: {NUM_STEPS}; final deployment is Full step {FINAL_FULL_STEP}.
LoRA weights are merged before export. No additional fitting for zero_shot.
24h forecast on 6h grid: VCD, viability, glucose, lactate, ammonium.
## Source and license
Uhlendorff et al. (2026), Dataset Based on Chinese Hamster Ovary (CHO) Cultivations
including Turbidity, Permittivity, O2 and CO2 Measurements, v1.
https://doi.org/10.5281/zenodo.20829178 . Source data: CC-BY-4.0.
Changes: causal right-edge binning; limited past-only fill for inline covariates.
Original data is not bundled. Code/model and source-data licenses are separate.
## Test results
{markdown_table(detail[(detail.method==method)&(detail.scope=="fed-batch")])}
## Limitations
Single-seed, small experiment-group holdout. Paired reactors stay in one split.
No Keytruda production data and no validated transfer to 2000L.
No titer, HCP, DNA or product-quality labels. No optimal harvest-day prediction.
Deviation flags are not contamination diagnoses. Harvest review threshold is illustrative.
Not validated for manufacturing control or batch release.
## Reproduce and infer
See preprocessing.json, split.json, requirements.txt, environment-lock.txt and cho_model.py.
"""
    (folder/"README.md").write_text(card)

cols=["method","selected_step","passes_viability_guard","mean_normalized_mae","mean_coverage_80","trainable_parameters",
      "train_seconds","train_peak_allocated_gib","ms_per_forecast_window","final_deployment_model"]
overview=markdown_table(summary[summary.scope=="all"][cols])
fed=markdown_table(summary[summary.scope=="fed-batch"][cols])
per_target=markdown_table(detail[detail.scope=="fed-batch"][["method","target","n","mae","rmse","coverage_80","improvement_vs_zero_shot_pct"]])
readme=f"""# CHO cultivation forecasting benchmark
Zero-shot vs LoRA vs Full fine-tuning on the same {BASE_MODEL} revision.
## Overall test comparison
{overview}
## Fed-batch test comparison
{fed}
## Fed-batch results by target
{per_target}
## How to read the tables
MAE/RMSE: lower is better. VCD: cells/mL; viability: percentage points;
glucose/lactate: g/L; ammonium: mM. Coverage target is 0.80, assessed with interval width.
Normalized MAE divides each target MAE by persistence MAE before averaging targets.
Zero-shot improvement = 100*(1 - method_MAE / zero_shot_MAE); negative means worse.
Final deployment: **Full step {FINAL_FULL_STEP}**, fixed by the user after prior Validation inspection.
The full training trajectory continues to {NUM_STEPS} steps. The exported Full weights are from step 350.
LoRA exports its Validation-selected checkpoint. Automatic suggestion is recorded separately.
Full 350 viability guard passed this run: **{bool(CHECKPOINT_CHOICES["full"]["eligible"])}**.
Training seconds include fit-time validation/checkpoints. Zero-shot training time is zero.
GPU values measure peak PyTorch allocated GiB, not device total. fit retains a source-model copy.
Inference time is warmed-up backtest wall time per window and includes DataFrame overhead.
## Protocol
Same base revision, targets, splits, context, horizon, steps, batch, seed and starting learning rate.
LoRA and Full each start from the original base. Full does not start from LoRA.
This is a same-setting single-seed comparison, not an equal-compute or tuned-optimum benchmark.
Every 50 steps: per-target MAE/RMSE, interval coverage/width, overall eval_loss and LR on fixed Validation windows.
Viability guard is reported for overall and Fed-batch Validation MAE; it does not override the user-fixed Full 350 choice. See results/deployment_selection.json and results/checkpoint_choices.json.
Paired reactors share the experiment split. Offline targets stay missing unless measured;
no future interpolation. See results/split.json and experiment_config.json.
Fed-batch Test contains only EXP012 (two reactors); avoid broad performance claims.
## Run on Colab
Upload your original Dataset.zip to MyDrive/CHO_AI/raw/Dataset.zip.
Open CHO_Colab_Drive_Training.ipynb in Colab, set GPU and ROOT, run cells in order.
The notebook generates this release directory and three model folders.
## Simulator inference
Install requirements.txt. Download a model folder from Hugging Face.
The final model is the Hugging Face **full** repository, exported from step 350.
Use cho_model.load_predict(model_dir, history_dataframe).
History must follow preprocessing.json with matching units and causal 6h grid.
## Data and license
Uhlendorff et al. (2026), https://doi.org/10.5281/zenodo.20829178 , CC-BY-4.0.
Changes: 6h right-edge last-observation binning and limited causal inline fill.
Raw data is excluded. Model and code: Apache-2.0; preserve upstream notices.
## Limits
No Keytruda clone/2000L validation. No titer or product-quality labels.
Cannot infer optimal harvest day or causal effects of arbitrary feed changes.
Single small holdout, one seed; no anomaly labels and no anomaly F1 claims.
Final claims require experiment-group cross-validation and repeated seeds.
Model uploads and GitHub uploads require explicitly enabling the corresponding cells.
"""
(REPO_DIR/"README.md").write_text(readme)
(REPO_DIR/"cho_model.py").write_text(CHO_MODULE)
(REPO_DIR/"requirements.txt").write_text(requirements)
(REPO_DIR/"environment-lock.txt").write_text(freeze)
(REPO_DIR/"experiment_config.json").write_text(json.dumps(environment,indent=2))
(REPO_DIR/"LICENSE").write_bytes(license_bytes)
for name,b in upstream.items(): (REPO_DIR/name).write_bytes(b)
(REPO_DIR/".gitignore").write_text(".env\n*.zip\nraw/\nmodels/\ncheckpoints/\n*.safetensors\n__pycache__/\n")
for name in ["comparison_summary.csv","comparison_by_target.csv","resources.csv","split.json"]:
    shutil.copy2(RUN/name,REPO_DIR/"results"/name)
# Include only aggregate diagnostics and figures in the GitHub release.
for name in ["validation_step_metrics.csv","checkpoint_choices.json","deployment_selection.json","checkpoint_selection_lora.csv","checkpoint_selection_full.csv"]:
    shutil.copy2(RUN/name,REPO_DIR/"results"/name)
(REPO_DIR/"results"/"relationships").mkdir(exist_ok=True)
for name in ["raw_vcd_viability_correlations.csv","raw_vcd_viability_scatter.png","raw_vcd_viability_timecourses.png"]:
    shutil.copy2(RUN/"relationships"/name,REPO_DIR/"results"/"relationships"/name)
for name in ["validation_viability_learning_curves.png","validation_all_target_mae_curves.png"]:
    shutil.copy2(RUN/name,REPO_DIR/"results"/name)
step_table=step_metrics[(step_metrics.scope=="fed-batch")&(step_metrics.phase=="all")&(step_metrics.target=="viability_pct")][["method","step","n","mae","rmse","coverage_80","mean_interval_width","eval_loss","learning_rate"]]
readme+="\n## Viability Validation at each checkpoint\n"+markdown_table(step_table)+"\n"
readme+="\n## Validation learning curves\n![Viability](results/validation_viability_learning_curves.png)\n![All targets](results/validation_all_target_mae_curves.png)\n"
readme+="\n## VCD and viability relationships\nRaw observed Train/Validation pairs only. Culture/phase Pearson and Spearman, within-culture centering, change-rate correlations; descriptive only, no causal or gradient-conflict claim.\n![Relationship](results/relationships/raw_vcd_viability_scatter.png)\n"
readme+="\nRepeated Validation tuning requires a new independent final evaluation; the previously inspected EXP012 Test is exploratory.\n"
(REPO_DIR/"README.md").write_text(readme)
(RUN/"comparison.md").write_text(readme)
from IPython.display import Markdown,display
display(Markdown(readme))
print("GitHub release:",REPO_DIR)


## 10. 실행 노트북을 GitHub 패키지에 포함


In [ ]:
NOTEBOOK_ON_DRIVE=ROOT/"CHO_Training.ipynb"  # 이 노트북 파일을 이 경로에 저장
if NOTEBOOK_ON_DRIVE.exists():
    nb=json.loads(NOTEBOOK_ON_DRIVE.read_text())
    for cell in nb["cells"]:
        if cell["cell_type"]=="code": cell["outputs"]=[];cell["execution_count"]=None
    nb["metadata"].pop("widgets",None)
    (REPO_DIR/"CHO_Training.ipynb").write_text(json.dumps(nb,ensure_ascii=False,indent=1))
else:
    print("GitHub 업로드 전에 NOTEBOOK_ON_DRIVE에 현재 노트북을 저장하세요.")


# 배포용

## 11. Hugging Face 배포

In [ ]:
UPLOAD_HF=True
HF_NAMESPACE="code7monkey"
HF_PREFIX="cho-state-chronos2-final350"
HF_PRIVATE=True
HF_LINKS={}
if UPLOAD_HF:
    assert HF_NAMESPACE!="YOUR_HF_USERNAME"
    from huggingface_hub import HfApi
    from getpass import getpass
    api=HfApi(token=getpass("Hugging Face write token: "))
    for method in METHODS:
        repo=f"{HF_NAMESPACE}/{HF_PREFIX}-{method.replace('_','-')}"
        api.create_repo(repo_id=repo,private=HF_PRIVATE,exist_ok=True)
        assert method!="full" or CHECKPOINT_CHOICES[method]["step"]==350
        api.upload_folder(repo_id=repo,folder_path=str(MODEL_DIRS[method]))
        api.upload_file(repo_id=repo,path_or_fileobj=str(RUN/"deployment_selection.json"),path_in_repo="deployment_selection.json")
        api.upload_file(repo_id=repo,path_or_fileobj=str(RUN/"comparison_summary.csv"),path_in_repo="comparison_summary.csv")
        HF_LINKS[method]="https://huggingface.co/"+repo
    links="\n".join(f"- {m}: {u}" for m,u in HF_LINKS.items())
    (REPO_DIR/"README.md").write_text(readme+"\n## Hugging Face models\n"+links+"\n")
    # Portable example for simulator integration. No Drive dependency.
    final_repo=HF_LINKS["full"].replace("https://huggingface.co/","")
    inference_example="""from huggingface_hub import snapshot_download
import sys
import pandas as pd
folder = snapshot_download(repo_id=MODEL_REPO)  # private repo requires HF_TOKEN/login
sys.path.insert(0, folder)
from cho_model import load_predict
history = pd.read_csv("history.csv", parse_dates=["timestamp"])
predictions = load_predict(folder, history)
print(predictions)
""".replace("MODEL_REPO",repr(final_repo))
    (REPO_DIR/"infer_hf.py").write_text(inference_example)
    del api
    print("Final model (Full 350):",HF_LINKS["full"])
    print(HF_LINKS)
else:
    print("Hugging Face 업로드 건너뜀")


## 12. GitHub 배포


In [ ]:
UPLOAD_GITHUB=True
GITHUB_REPO_NAME="cho-cultivation-chronos2-final350"
GITHUB_PRIVATE=True
if UPLOAD_GITHUB:
    import base64,urllib.error,urllib.parse
    from getpass import getpass
    assert (REPO_DIR/"CHO_Training.ipynb").exists(), "10번 셀에서 노트북을 포함하세요."
    gh_token=getpass("GitHub token: ")
    def gh_request(method,path,payload=None):
        body=None if payload is None else json.dumps(payload).encode()
        req=urllib.request.Request("https://api.github.com"+path,data=body,method=method,
            headers={"Authorization":"Bearer "+gh_token,"Accept":"application/vnd.github+json",
                     "Content-Type":"application/json","X-GitHub-Api-Version":"2026-03-10"})
        try:
            with urllib.request.urlopen(req,timeout=60) as r:return json.load(r)
        except urllib.error.HTTPError as e:
            raise RuntimeError(f"GitHub HTTP {e.code}: 저장소 이름/권한/부분 업로드 상태를 확인하세요.") from None
    try:
        me=gh_request("GET","/user")["login"]
        repo=gh_request("POST","/user/repos",dict(name=GITHUB_REPO_NAME,private=GITHUB_PRIVATE,
            description="CHO forecasting: zero-shot vs LoRA vs full fine-tuning",auto_init=False))
        files=sorted(f for f in REPO_DIR.rglob("*") if f.is_file())
        for f in files:
            rel=f.relative_to(REPO_DIR).as_posix()
            assert f.suffix not in {".safetensors",".zip"}
            gh_request("PUT",f"/repos/{me}/{GITHUB_REPO_NAME}/contents/"+urllib.parse.quote(rel),
                dict(message="Add CHO benchmark: "+rel,
                     content=base64.b64encode(f.read_bytes()).decode()))
        print("GitHub:",repo["html_url"])
    finally:
        gh_token=""
else:
    print("GitHub 업로드 건너뜀. 결과는 Drive의 github_release 폴더에 있습니다.")
